## Where behavior, ripple, parsed replay data lives

In [1]:
%reload_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
import numpy as np

### 1. Animal's behavior + annotated future, past, past reward arms
These data live in the table called ```TrialChoice()```.

You will need to have Shijie's table declaration script ```Analysis_SGU.py``` under a folder called ```shijiegu```.


In [3]:
from spyglass.shijiegu.Analysis_SGU import TrialChoice

# helper functions
from spyglass.utils.nwb_helper_fn import get_nwb_copy_filename

[2025-10-27 14:19:27,291][INFO]: DataJoint 0.14.4 connected to shijiegu-alt@lmf-db.cin.ucsf.edu:3306


Our datasets are organized in units of 
- the name of the rat
- the day on which recording happened.

The 5 rats I have are Molly, Eliot, Julio, Klein, and Lewis. The dates are listed below:
- Molly: 20220415,20220416,20220417,20220418,20220419,20220420
- Eliot: 20221018,20221019,20221020,20221021,20221022,20221023,20221024,20221025,20221026
- Julio: 20230731,20230801,20230802,20230803,20230804,20230805,20230806,20230807,20230808,20230809,20230810,20230811
- Klein: 20231101,20231102,20231103,20231104,20231105,20231106,20231107,20231108,20231109,20231111
- Lewis: 20240105, 20240106, 20240107,20240108,20240109,20240110,20240113,20240114


In [4]:
'''choose a day'''
nwb_file_name = 'molly20220418.nwb' # animal name + data + .nwb

# some idiosynchrocy about Frank Lab database.
# The idea is that large raw ephys lives in nwb_file_name and the rest lives in "nwb_copy_file_name"
# So to get any processed data, use nwb_copy_file_name, which just adds an underscore after the date.
nwb_copy_file_name = get_nwb_copy_filename(nwb_file_name)  #'molly20220418_.nwb'

# all the maze sessions/epochs on this day
all_epochs=list((TrialChoice() & 
                 {'nwb_file_name':nwb_copy_file_name}).fetch('epoch'))
print('epochs all this day:',all_epochs)

epochs all this day: [2, 4, 6, 8, 10]


In [5]:
pd.set_option('display.max_rows', None)

In [6]:
'''choose an epoch'''
epoch_num = 4

In [7]:
'''get the behavior on this day and this epoch'''
T=pd.DataFrame((TrialChoice() & {'nwb_file_name':nwb_copy_file_name,
                                 'epoch':epoch_num}).fetch1('choice_reward'))
T

,timestamp_H,Home,timestamp_O,OuterWellIndex,rewardNum,current,future_H,future_O,past,past_reward
1,1.650308e+09,1.0,1.650308e+09,4.0,1.0,4.0,4.0,2.0,NaN,NaN
2,1.650308e+09,1.0,1.650308e+09,2.0,1.0,2.0,2.0,3.0,4.0,NaN
3,1.650308e+09,1.0,1.650308e+09,3.0,2.0,3.0,3.0,1.0,2.0,NaN
4,1.650308e+09,1.0,1.650308e+09,1.0,1.0,1.0,1.0,4.0,3.0,3.0
5,1.650308e+09,1.0,1.650308e+09,4.0,2.0,4.0,4.0,2.0,1.0,3.0
6,1.650308e+09,1.0,1.650308e+09,2.0,2.0,2.0,2.0,3.0,4.0,4.0
7,1.650308e+09,1.0,1.650308e+09,3.0,1.0,3.0,3.0,1.0,2.0,2.0
8,1.650308e+09,1.0,1.650308e+09,1.0,2.0,1.0,1.0,4.0,3.0,2.0
9,1.650308e+09,1.0,1.650308e+09,4.0,1.0,4.0,4.0,2.0,1.0,1.0
10,1.650308e+09,1.0,1.650308e+09,2.0,1.0,2.0,2.0,1.0,4.0,1.0


#### Notes:
**timestamp_H**: Unix time for poking the home well.
Home: if the animal poked the home well at that trial. Animal is pretrained to do that to initiate a trial. Feel free to ignore the few trials that the animal skipped the home. I call these trials "illegal trials".

**timestamp_O**: Unix time for poking the outer well.

**OuterWellIndex**: The outer well/arm choice animal makes.

**rewardNum**: 0 - illegal trial; 1 - incorrect trial; 2 - correct trial

The rest can be derived from previous columns.

**current**: the same as OuterWellIndex, can ignore.

**future_H**: the same as OuterWellIndex, can ignore.

**future_O**: the arm the rat will choose on the next trial.

**past**: the arm the rat picked on the last trial.

**past_reward**: the arm the rat picked on the last trial, and in addition, he got a reward there.

### Work in progress from below.

### 2. Animal position data.
You might need this or might not.

In [8]:
from spyglass.common.common_position import IntervalPositionInfo, RawPosition, IntervalLinearizedPosition, TrackGraph

### 3. Spiking data

### 4. Replay data
Coming soon, I need to push the dataframes into the database in a new format for all rats. But it will look like this:

In [9]:
from spyglass.shijiegu.load import load_run_sessions
from spyglass.shijiegu.Analysis_SGU import RippleTimesWithDecode

In [10]:
# Find all sessions on this day
run_session_ids, run_session_names, pos_session_names = load_run_sessions(nwb_copy_file_name)

*nwb_file_name *epoch    epoch_name     position_inter
+------------+ +-------+ +------------+ +------------+
molly20220418_ 1         01_Seq2Sleep1  pos 0 valid ti
molly20220418_ 2         02_Seq2Session pos 1 valid ti
molly20220418_ 3         03_Seq2Sleep2  pos 2 valid ti
molly20220418_ 4         04_Seq2Session pos 3 valid ti
molly20220418_ 5         05_Seq2Sleep3  pos 4 valid ti
molly20220418_ 6         06_Seq2Session pos 5 valid ti
molly20220418_ 7         07_Seq2Sleep4  pos 6 valid ti
molly20220418_ 8         08_Seq2Session pos 7 valid ti
molly20220418_ 9         09_Seq2Sleep5  pos 8 valid ti
molly20220418_ 10        10_Seq2Session pos 9 valid ti
molly20220418_ 11        11_Seq2Sleep6  pos 10 valid t
 (Total: 11)



In [11]:
# pick a session
session_name = run_session_names[0]
print(session_name)

02_Seq2Session1


In [12]:
query = {'nwb_file_name': nwb_copy_file_name,
         'interval_list_name': session_name,
         "classifier_param_name":"default_decoding_gpu_4armMaze",
         "encoding_set":"2Dheadspeed_above_4",
         "decode_threshold_method":"MUA_M05SD"}
path_to_ripple_times = (RippleTimesWithDecode & query).fetch1("ripple_times")

try: # old format for data processed later
    ripple_times = pd.DataFrame(path_to_ripple_times)
except: # new format for data processed later
    ripple_times = pd.read_pickle(path_to_ripple_times)

### for each ripple event, I have parsed
- the trial in which it occured
- the start time, end time of each ripple
- the continuous interval(s) within the ripple
- the fragmented interval(s) within the ripple
- and other statistics about that ripple

In [13]:
ripple_times

,event_number,start_time,animal_location,trial_number,cont_intvl,frag_intvl,cont_intvl_replay,end_time,duration,mean_zscore,median_zscore,max_zscore,min_zscore,speed_at_start,speed_at_end,max_speed,min_speed,median_speed,mean_speed
0,1,1.650303e+09,well4,1,"[[1650303252.936586, 1650303253.006586]]",[],[[]],1.650303e+09,0.071000,147.215883,158.129299,218.404138,82.185485,1.719982e+00,3.405536e+00,3.405536e+00,1.719982e+00,2.587314e+00,2.577609e+00
1,2,1.650303e+09,well4,1,"[[1650303254.1365852, 1650303254.298585]]",[],[[4]],1.650303e+09,0.165000,126.656481,122.053295,183.785176,85.441928,3.995007e+00,3.992519e+00,3.995007e+00,1.258586e+00,2.547996e+00,2.482513e+00
2,3,1.650303e+09,well2,1,"[[1650303261.1885786, 1650303261.2765784]]",[],[[2]],1.650303e+09,0.089000,142.311060,135.305319,211.987384,84.171689,3.982026e+00,1.598308e+00,3.982026e+00,1.598308e+00,2.539862e+00,2.625220e+00
3,4,1.650303e+09,well2,1,"[[1650303261.706578, 1650303261.742578]]",[],[[2]],1.650303e+09,0.037000,150.140898,159.832319,201.986928,82.264354,2.962627e+00,3.081341e+00,3.081341e+00,2.962627e+00,3.052335e+00,3.048279e+00
4,5,1.650303e+09,arm2,1,"[[1650303272.372568, 1650303272.538568]]",[],[[2]],1.650303e+09,0.169000,166.984238,146.755207,312.191901,88.889942,3.978831e+00,3.995354e+00,3.995354e+00,1.937071e+00,2.583384e+00,2.689648e+00
5,6,1.650303e+09,home,3,"[[1650303280.1345608, 1650303280.2345607]]",[],[[0]],1.650303e+09,0.101000,153.282340,118.802612,316.836065,82.249822,3.973340e+00,1.257277e+00,3.973340e+00,1.257277e+00,2.570525e+00,2.582997e+00
6,7,1.650303e+09,well3,3,"[[1650303293.4465485, 1650303293.4965484]]","[[1650303293.4145484, 1650303293.4445484]]",[[3]],1.650303e+09,0.105000,128.103350,123.202559,204.418017,81.382671,1.235064e+00,2.444325e-01,1.235064e+00,2.444325e-01,6.385950e-01,6.747193e-01
7,8,1.650303e+09,well3,3,"[[1650303293.7365482, 1650303293.8225482]]",[],[[3]],1.650303e+09,0.088000,129.819578,114.717974,239.043078,81.267749,5.162954e-01,3.630146e-01,5.162954e-01,3.585136e-01,3.777372e-01,4.068452e-01
8,9,1.650303e+09,well3,3,"[[1650303294.2185478, 1650303294.2805476]]",[],[[3]],1.650303e+09,0.065000,137.941324,148.005203,224.976581,82.124282,2.137516e+00,2.138567e+00,2.205644e+00,2.137516e+00,2.171794e+00,2.171811e+00
9,10,1.650303e+09,well3,3,"[[1650303294.6605473, 1650303294.7425473]]",[],[[3]],1.650303e+09,0.083000,131.804506,115.990104,240.983384,81.561554,1.195341e+00,1.324755e+00,1.324755e+00,9.653943e-01,1.083308e+00,1.101802e+00


### 5. Decode data